# 02 · Zadanie I1: Cellpose-SAM w trybie inferencji na MoNuSeg

Porównujemy dwa modele z maskami ground truth (GT) z MoNuSeg:

- **A — wyrocznia**: w pełni wytrenowany Cellpose-SAM `cpsam` (wersja z kwietnia 2025, wagi przypięte
  SHA-256 w `models/oracle/cpsam`). W symulacji AL to on „adnotuje” obrazy.
- **B — niewytrenowany Cellpose-SAM**: ta sama architektura, encoder z oryginalnych wag SAM ViT-L
  (`models/student_init/sam_vit_l_0b3195.pth`), losowa głowica wyjściowa. Nigdy nie widział komórek.
  To dokładnie punkt startowy studenta w zadaniu I3.

Ewaluacja na **14 obrazach testowych** (wyłącznie ewaluacja: test nie wpływa na żaden wybór) oraz na
**7 obrazach val**. Wynik wyroczni na val jest poprzeczką dla studenta w I3.

**Zastrzeżenie:** MoNuSeg jest w danych treningowych Cellpose-SAM (Pachitariu i in. 2025), więc wynik A
na MoNuSeg nie jest niezależny i jest prawdopodobnie zawyżony.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cellpose import io as cp_io
from skimage.segmentation import find_boundaries

from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import monuseg
from histopathology_pnw.device import get_device
from histopathology_pnw.inference import EVAL_KWARGS, predict
from histopathology_pnw.metrics import evaluate_all, summarize
from histopathology_pnw.models.oracle import load_oracle, oracle_masks
from histopathology_pnw.models.student import build_student_init, load_student

cp_io.logger_setup()
device = get_device()
out_dir = Settings().outputs_dir / "i1"
out_dir.mkdir(parents=True, exist_ok=True)
print("device:", device, "| wyniki:", out_dir)

In [ ]:
split = monuseg.load_train_val_split()
by_id = monuseg.samples_by_id("train")
sets = {
    "test": monuseg.list_split("test"),
    "val": [by_id[i] for i in split["val"]],
}
images = {name: [monuseg.read_image(s.image_path) for s in ss] for name, ss in sets.items()}
gts = {name: [monuseg.read_mask(s) for s in ss] for name, ss in sets.items()}
ids = {name: [s.image_id for s in ss] for name, ss in sets.items()}
{name: len(ss) for name, ss in sets.items()}

## Metryki

Dopasowanie instancji: predykcja i jądro GT pasują, gdy IoU > 0.5 (przy tym progu dopasowanie jest
jednoznaczne). Z dopasowań TP, a niedopasowane to FP (predykcje) i FN (jądra GT).

| metryka | definicja | uwagi |
|---|---|---|
| **PQ** (= PQ@0.5) | SQ × DQ | Kirillov i in. 2019; standard w segmentacji jąder (HoVer-Net) |
| SQ | średnie IoU dopasowanych par | jakość konturów |
| DQ | TP / (TP + ½FP + ½FN) | to samo co F1@0.5: jakość detekcji |
| PQ@0.75 | PQ z progiem IoU 0.75 | ostrzejsze wymaganie na kontury |
| AJI | aggregated Jaccard index | metryka oryginalnego MoNuSeg (Kumar i in. 2017) |
| AP@0.5 | TP / (TP + FP + FN) | „average precision” w konwencji Cellpose |

Średnie liczone po obrazach (tak raportuje się MoNuSeg). `pq_pooled` liczy PQ z sumy TP/FP/FN całego zbioru.
Ustawienia inferencji są identyczne dla obu modeli:

In [ ]:
EVAL_KWARGS

## A — wyrocznia `cpsam`

`load_oracle()` sprawdza SHA-256 pliku przed załadowaniem (Cellpose przy złej ścieżce po cichu ładuje
`cpsam_v2`). Predykcje są zapisywane w `outputs/oracle_masks/monuseg/`, więc liczą się tylko raz.
Z tego cache korzysta też notebook 03.

In [ ]:
oracle = load_oracle(device)
preds_a = {name: oracle_masks(ss, model=oracle) for name, ss in sets.items()}

## B — niewytrenowany Cellpose-SAM (encoder SAM + losowa głowica)

`build_student_init` wgrywa wagi encodera SAM ViT-L z tą samą adaptacją, którą robi `CPSAM.__init__`
(patch 16 → 8 px, pozycje 64×64 → 32×32), z `strict=True`, czyli bez brakujących ani nadmiarowych kluczy.
Głowica wyjściowa jest losowa (seed 0). Spodziewamy się, że model nie znajdzie prawie nic: encoder
zna obrazy naturalne, ale nikt go nie nauczył, jak z cech zrobić flow fields Cellpose.

In [ ]:
student_init = build_student_init(
    seed=0, out_path=Settings().outputs_dir / "student_init" / "seed0.pth"
)
untrained = load_student(student_init, device)
preds_b = {name: predict(untrained, images[name]) for name in sets}

## Wyniki

In [ ]:
tables, summary = {}, []
for name in sets:
    for model, preds in (("A_oracle", preds_a), ("B_untrained", preds_b)):
        table = evaluate_all(gts[name], preds[name], ids[name]).assign(set=name, model=model)
        tables[(name, model)] = table
        summary.append({"set": name, "model": model, **summarize(table)})
per_image = pd.concat(tables.values(), ignore_index=True)
summary = pd.DataFrame(summary)
per_image.to_csv(out_dir / "metrics_per_image.csv", index=False)
summary.to_csv(out_dir / "metrics_summary.csv", index=False)

oracle_val = summary[(summary["set"] == "val") & (summary["model"] == "A_oracle")].iloc[0].to_dict()
(out_dir / "oracle_val_summary.json").write_text(json.dumps(oracle_val, indent=2))

summary.set_index(["set", "model"]).round(4)

In [ ]:
cols = ["image_id", "pq", "sq", "dq", "aji", "n_gt", "n_pred", "tp", "fp", "fn"]
tables[("test", "A_oracle")][cols].round(3)

In [ ]:
t = tables[("test", "A_oracle")].sort_values("pq")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].barh(t["image_id"], t["pq"])
ax[0].set_xlabel("PQ (wyrocznia, test)")
ax[1].scatter(t["n_gt"], t["n_pred"])
lim = [0, max(t["n_gt"].max(), t["n_pred"].max()) * 1.05]
ax[1].plot(lim, lim, "k--", lw=1)
ax[1].set_xlabel("liczba jąder w GT")
ax[1].set_ylabel("liczba jąder wykrytych przez wyrocznię")
plt.tight_layout()

## Wizualizacja: GT vs A vs B

Najlepszy, mediana i najgorszy obraz testowy według PQ wyroczni, wycinek 300×300 px. Kontury: zielone = GT,
żółte = predykcja.

In [ ]:
def contours(image, mask, color):
    out = image.copy()
    out[find_boundaries(mask, mode="inner")] = color
    return out


order = t["image_id"].tolist()
picks = [order[-1], order[len(order) // 2], order[0]]
crop = np.s_[350:650, 350:650]
fig, ax = plt.subplots(len(picks), 4, figsize=(18, 4.6 * len(picks)))
for row, image_id in enumerate(picks):
    k = ids["test"].index(image_id)
    img = images["test"][k][crop]
    panels = [
        (img, "obraz"),
        (contours(img, gts["test"][k][crop], (0, 255, 0)), "GT"),
        (contours(img, preds_a["test"][k][crop], (255, 255, 0)), "A: wyrocznia"),
        (contours(img, preds_b["test"][k][crop], (255, 255, 0)), "B: niewytrenowany"),
    ]
    pq = t.set_index("image_id").loc[image_id, "pq"]
    for col, (panel, title) in enumerate(panels):
        ax[row, col].imshow(panel)
        ax[row, col].set_title(f"{title}" + (f"\n{image_id}, PQ_A={pq:.3f}" if col == 0 else ""))
        ax[row, col].axis("off")
plt.tight_layout()
fig.savefig(out_dir / "gt_vs_a_vs_b.png", dpi=120)

## Dyskusja

Wyniki z przebiegu na Macu (MPS, 2026-10-15); na CUDA mogą się różnić w trzecim miejscu po przecinku.

| zbiór | model | PQ | SQ | DQ (= F1@0.5) | PQ@0.75 | AJI |
|---|---|---|---|---|---|---|
| test (14) | A: wyrocznia | 0.654 | 0.757 | 0.863 | 0.425 | 0.690 |
| test (14) | B: niewytrenowany | 0 | 0 | 0 | 0 | 0 |
| val (7) | A: wyrocznia | 0.644 | 0.785 | 0.821 | 0.475 | 0.701 |
| val (7) | B: niewytrenowany | 0 | 0 | 0 | 0 | 0 |

- Wyrocznia dobrze **wykrywa** jądra (DQ ≈ 0.86), słabiej odtwarza **kontury** (SQ ≈ 0.76, a przy progu IoU 0.75 PQ spada do 0.43).
  Część tej różnicy to konwencja adnotacji: ręczne poligony MoNuSeg vs granice, których Cellpose-SAM nauczył się na wielu zbiorach.
- Na teście wyrocznia daje ~2× więcej FP niż FN (1246 vs 620): raczej dzieli jądra albo znajduje jądra nieoznaczone w GT, niż je gubi.
- Wynik A jest zawyżony, bo MoNuSeg był w danych treningowych Cellpose-SAM.
- B nie znajduje ani jednego jądra (0 predykcji na 21 obrazach). Losowa głowica nie daje sensownych flow fields,
  więc cała wiedza o jądrach w studencie (I3) będzie pochodzić z obrazów wybranych w AL.

### Na co patrzeć przy własnym uruchomieniu

- **A vs GT**: DQ (detekcja) zwykle jest wyraźnie wyższe niż SQ (kontury), a PQ@0.75 dużo niższe niż PQ.
  Wyrocznia dobrze znajduje jądra, gorzej odtwarza dokładne granice ręcznych poligonów. Na wykresie
  rozrzutu widać, czy wyrocznia systematycznie wykrywa za dużo (FP) czy za mało (FN) jąder.
- **A jest zawyżone**: MoNuSeg był w danych treningowych Cellpose-SAM. Liczby trzeba raportować z tym zastrzeżeniem.
- **B ≈ 0**: encoder SAM bez wytrenowanej głowicy nie produkuje sensownych flow fields, więc nie powstają maski.
  To pokazuje, że cała wiedza o jądrach w studencie (I3) będzie pochodzić z obrazów wybranych w AL.
- `oracle_val_summary.json` zawiera PQ wyroczni na val: to poprzeczka dla notebooka 03.